[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/state-schema.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239426-lesson-1-state-schema)

# 状态模式

## 回顾

在模块1中，我们打下了基础！我们构建了一个可以执行以下操作的agent：

* `act` - 让模型调用特定工具
* `observe` - 将工具输出传回模型
* `reason` - 让模型对工具输出进行推理，决定下一步做什么（例如，调用另一个工具或直接响应）
* `persist state` - 使用内存检查点保存器支持具有中断的长时间运行对话
 
而且，我们展示了如何在LangGraph Studio中本地提供服务或使用LangGraph Cloud进行部署。

## 目标

在这个模块中，我们将建立对状态和记忆的更深入理解。

首先，让我们回顾定义状态模式的几种不同方法。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph

## 模式

当我们定义LangGraph `StateGraph`时，我们使用[状态模式](https://langchain-ai.github.io/langgraph/concepts/low_level/#state)。

状态模式表示我们的图将使用的数据的结构和类型。

所有节点都应该与该模式通信。

LangGraph在定义状态模式方面提供了灵活性，适应各种Python[类型](https://docs.python.org/3/library/stdtypes.html#type-objects)和验证方法！

## TypedDict

正如我们在模块1中提到的，我们可以使用Python的`typing`模块中的`TypedDict`类。

它允许您指定键及其相应的值类型。
 
但是，请注意这些是类型提示。

它们可以被静态类型检查器（如[mypy](https://github.com/python/mypy)）或IDE使用，在代码运行之前捕获潜在的类型相关错误。

但它们在运行时不被强制执行！

In [ ]:
from typing_extensions import TypedDict

class TypedDictState(TypedDict):
    foo: str
    bar: str

对于更具体的值约束，您可以使用像`Literal`类型提示这样的功能。

在这里，`mood`只能是"happy"或"sad"。

In [ ]:
from typing import Literal

class TypedDictState(TypedDict):
    name: str
    mood: Literal["happy","sad"]

我们可以在LangGraph中使用我们定义的状态类（例如，这里的`TypedDictState`），只需将其传递给`StateGraph`。

而且，我们可以将每个状态键视为图中的一个"通道"。

如模块1中所讨论的，我们在每个节点中覆盖指定键或"通道"的值。

In [ ]:
import random
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

def node_1(state):
    print("---节点 1---")
    return {"name": state['name'] + " 是 ... "}

def node_2(state):
    print("---节点 2---")
    return {"mood": "happy"}

def node_3(state):
    print("---节点 3---")
    return {"mood": "sad"}

def decide_mood(state) -> Literal["node_2", "node_3"]:
        
    # 在这里，让我们在节点2、3之间进行50/50分割
    if random.random() < 0.5:

        # 50%的时间，我们返回节点2
        return "node_2"
    
    # 50%的时间，我们返回节点3
    return "node_3"

# 构建图
builder = StateGraph(TypedDictState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_conditional_edges("node_1", decide_mood)
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

因为我们的状态是一个字典，我们只需用字典调用图来设置状态中`name`键的初始值。

In [ ]:
graph.invoke({"name":"Lance"})

## Dataclass

Python的[dataclasses](https://docs.python.org/3/library/dataclasses.html)提供了[定义结构化数据的另一种方法](https://www.datacamp.com/tutorial/python-data-classes)。

Dataclass为创建主要用于存储数据的类提供了简洁的语法。

In [ ]:
from dataclasses import dataclass

@dataclass
class DataclassState:
    name: str
    mood: Literal["happy","sad"]

要访问`dataclass`的键，我们只需要修改`node_1`中使用的下标：

* 我们对`dataclass`状态使用`state.name`，而不是对上面的`TypedDict`使用`state["name"]`

您会注意到一些有点奇怪的事情：在每个节点中，我们仍然返回一个字典来执行状态更新。
 
这是可能的，因为LangGraph分别存储状态对象的每个键。

节点返回的对象只需要具有与状态中匹配的键（属性）！

在这种情况下，`dataclass`有键`name`，所以我们可以通过从我们的节点传递一个字典来更新它，就像我们在状态是`TypedDict`时所做的那样。

In [ ]:
def node_1(state):
    print("---节点 1---")
    return {"name": state.name + " 是 ... "}

# 构建图
builder = StateGraph(DataclassState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_conditional_edges("node_1", decide_mood)
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

我们使用`dataclass`调用来设置状态中每个键/通道的初始值！

In [ ]:
graph.invoke(DataclassState(name="Lance",mood="sad"))

## Pydantic

如前所述，`TypedDict`和`dataclasses`提供类型提示，但它们不在运行时强制执行类型。
 
这意味着您可能会分配无效值而不会引发错误！

例如，我们可以将`mood`设置为`mad`，即使我们的类型提示指定`mood: list[Literal["happy","sad"]]`。

In [ ]:
dataclass_instance = DataclassState(name="Lance", mood="mad")

[Pydantic](https://docs.pydantic.dev/latest/api/base_model/)是一个使用Python类型注释进行数据验证和设置管理的库。

由于其验证功能，它特别[适合在LangGraph中定义状态模式](https://langchain-ai.github.io/langgraph/how-tos/state-model/)。

Pydantic可以执行验证，在运行时检查数据是否符合指定的类型和约束。

In [ ]:
from pydantic import BaseModel, field_validator, ValidationError

class PydanticState(BaseModel):
    name: str
    mood: str # "happy" 或 "sad"

    @field_validator('mood')
    @classmethod
    def validate_mood(cls, value):
        # 确保mood是"happy"或"sad"
        if value not in ["happy", "sad"]:
            raise ValueError("每个mood必须是'happy'或'sad'")
        return value

try:
    state = PydanticState(name="John Doe", mood="mad")
except ValidationError as e:
    print("验证错误:", e)

我们可以在图中无缝地使用`PydanticState`。

In [ ]:
# 构建图
builder = StateGraph(PydanticState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_conditional_edges("node_1", decide_mood)
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke(PydanticState(name="Lance",mood="sad"))